## **Section 1: Programming Test:** 

See "Innoventes_DataEngg_Programming_Test_2023DSEP01.pdf" in PANDA folder. Section 1 has two problems defined around the sample CSV files below. Download the flowing
CSV file to solve the problems in Section1.
user_flags.csv
video.csv
video_review.csv
Write scripts/programs in python to do the following:
Problem 1
The data engineering team wants to clean the dataset user_flags.csv. The user_flags.csv.
Examine rows in the CSV file user_flags.csv, and create a new CSV file containing the row that
have missing values in more than one column

In [0]:
                     ## Important Note Before Solving Question ## 
# ✔ df.isnull()
# Creates True/False for missing values
# ✔ .sum(axis=1)
# Counts missing columns row-wise
# ✔ > 1
# Filters only rows with more than one missing column
# pandas: df.filter() selects columns by name patterns, not rows
# PySpark: df.filter() filters rows by conditions ✓

# ✅ What is index in pandas?   
# Every pandas DataFrame has a row index (like row numbers): 0,1,2 are indexes
#    user_first_name   user_last_name
# 0  John              Smith
# 1  Jane              Doe
# 2  Ella              NaN    
# ✅ index=False 👉 Means: ❌ Do NOT write the index into the CSV file
# filtered_df.to_csv("file.csv", index=False): 
# Output will look like: 👉
# user_first_name,user_last_name,description,flag_id
# Ella,, ,vWGufUE
# Jaylene,, ,PaOJ5H1
# ,, ,jHAa9Go
# index=True 👉 Means: ✔ Include index as a separate column
# filtered_df.to_csv("file.csv", index=True)
# ,index,user_first_name,user_last_name,description,flag_id
# 0,Ella,, ,vWGufUE
# 1,Jaylene,, ,PaOJ5H1
# 2,, , ,jHAa9Go


import pandas as pd
df = pd.read_csv("/Volumes/workspace/default/programming_test/user_flags.csv")
#checking missing value and how many nulls are in each row
df['missing_count'] = df.isnull().sum(axis=1)
display(df)
filter_df = df[df["missing_count"]>1]
display(filter_df)
filter_df.to_csv(("/Volumes/workspace/default/programming_test/user_flags_output.csv"))

In [0]:
# null_counts = [
#     when(col("first_name").isNull(), 1).otherwise(0),
#     when(col("last_name").isNull(), 1).otherwise(0),
#     when(col("flag_id").isNull(), 1).otherwise(0)
# ]
# df.withColumn("missing_count", sum(null_counts))

# df.withColumn("missing_count", 
#     when(col("first_name").isNull(), 1).otherwise(0) +
#     when(col("last_name").isNull(), 1).otherwise(0) +
#     when(col("flag_id").isNull(), 1).otherwise(0)
# )
from pyspark.sql.functions import col, sum as spark_sum, when

# Read CSV
df = spark.read.option("header", True).csv("/Volumes/workspace/default/programming_test/user_flags.csv")

# 2. Count missing values column-wise for each row
# We create a list of conditions: if a column is null, give it a 1, else 0. Then we sum them up.
null_counts = [when(col(c).isNull(), 1).otherwise(0).alias(c) for c in df.columns]
df_with_count = df.withColumn("missing_count", sum(null_counts))

display(df_with_count)

# Filter rows with more than 1 missing value
filtered_df = df_with_count.filter(col("missing_count") > 1)
display(filtered_df)

# Save to CSV (without the missing_count column if you want)
filtered_df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/programming_test/user_flags_output")

## **Problem 2**

Users can review videos and add comments. Users can review multiple videos and multiple
users can review the same video. Once the video is reviewed by the user, the review details are
captured in the video_review.csv.
Retrieve the user who has flagged the most distinct videos that end up reviewed as Yes in the
is_reviewed column.
Create a CSV file that has one column, showing the user’s full name. If multiple users have
reviewed the same number of videos, show the list of user names. In the user's full name,
include a space between the first and the last name.

In [0]:
# The Goal: Find the user(s) who reviewed the highest number of unique videos where the review status is "Yes", and output a file with just their full name.
# grouped_df.agg(F.max("video_count")) 
# +------------------+
# |max(video_count)  |
# +------------------+
# |        5         |
# +------------------+ 🔴 Problem You need: max_count = 5 ✅ Solution: .collect() and .collect()[0] Take first element from list and second .collect()[0][0] 👉 Take first value inside that row . Result 5
# concat_ws: ws = "with separator" and Automatically adds space between columns

from pyspark.sql.functions import *

# Step 1: Load CSV files
user_flags_df = spark.read.option("header", True).csv("/Volumes/workspace/default/programming_test/user_flags.csv")
video_review_df = spark.read.option("header", True).csv("/Volumes/workspace/default/programming_test/video_review.csv")

# Step 2: Filter only reviewed = Yes
video_review_filtered = video_review_df.filter(col("is_reviewed") == "Yes")

# Step 3: Join with user_flags on flag_id
merged_df = video_review_filtered.join(user_flags_df, on="flag_id", how="inner")

# Step 4: Count DISTINCT videos per user
grouped_df = merged_df.groupBy("user_first_name", "user_last_name").agg(countDistinct("video_id").alias("video_count"))
display(grouped_df)
# Step 5: Get max video count
max_count = grouped_df.agg(max("video_count")).collect()[0][0]

# Step 6: Filter users with max count
top_users = grouped_df.filter(col("video_count") == max_count)

# Step 7: Create full name (FIXED HERE ✅)
top_users = top_users.withColumn("full_name",concat_ws(" ", col("user_first_name"), col("user_last_name")))

# Step 8: Select required column
final_df = top_users.select("full_name")

# Step 9: Save output
final_df.write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/programming_test/top_users")

# Step 10: Display
display(final_df)

In [0]:
import requests

# 1. Define the API endpoint URL
url = "https://mail.google.com/"

# 2. Send a GET request to the API
response = requests.get(url)

# 3. Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # 4. Parse the response content as JSON
    data = response.json()
    print("Data retrieved successfully:")
    print(data)
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")
